# Explore raw MovieLens data

Quick look at the raw `movies.csv` / `ratings.csv` files before writing any cleaning logic, mirroring `explore_raw.ipynb` for the Amazon side. Key differences from the Amazon 2023 dump:
- Plain CSV with a header row, not `.jsonl.gz` — no gzip/JSON-per-line parsing needed.
- Ratings join key is `movieId` (renamed `item_id` downstream), and `timestamp` is **Unix seconds**, not milliseconds.
- `movies.csv` bakes the release year into the title string, e.g. `"Toy Story (1995)"` — needs a regex split, not a separate column.
- `genres` is a single pipe-delimited string, e.g. `"Adventure|Animation|Comedy"` (or the literal `"(no genres listed)"`), not a JSON list needing `eval()`.
- No review text, no price, no `parent_asin`/`asin` split — one row per (user, movie) rating, one row per movie.

# 0. Import

In [5]:
import pandas as pd
from local_package.config.data import MOVIELENS_RAW_DIR

# 1. Configuration

In [ ]:
# Path
DATA_DIR = MOVIELENS_RAW_DIR  # swap for any MovieLens release with the same movies.dat/ratings.dat schema

MOVIES_FILE = DATA_DIR / "movies.dat"
RATINGS_FILE = DATA_DIR / "ratings.dat"

# Column names defined since .dat files do not have headers
MOVIE_COLUMN_NAMES = ["MovieID", "Title", "Genres"]
RATINGS_COLUMN_NAMES = ["UserID", "MovieID", "Rating", "Timestamp"]

In [26]:
# Records peeking
N_PEEK = 3  # how many raw rows to pretty-print

## Raw record peek

In [27]:
print(f"--- first {N_PEEK} raw movies.dat lines ---")
with open(MOVIES_FILE) as f:
    for _, line in zip(range(N_PEEK + 1), f):  # +1 to include the header row
        print(line.rstrip())

--- first 3 raw movies.dat lines ---
1::Toy Story (1995)::Adventure|Animation|Children|Comedy|Fantasy
2::Jumanji (1995)::Adventure|Children|Fantasy
3::Grumpier Old Men (1995)::Comedy|Romance
4::Waiting to Exhale (1995)::Comedy|Drama|Romance


In [28]:
print(f"--- first {N_PEEK} raw ratings.csv lines ---")
with open(RATINGS_FILE) as f:
    for _, line in zip(range(N_PEEK + 1), f):
        print(line.rstrip())

--- first 3 raw ratings.csv lines ---
1::122::5::838985046
1::185::5::838983525
1::231::5::838983392
1::292::5::838983421


## Load into DataFrames

In [29]:
movies_df = pd.read_csv(
    MOVIES_FILE, 
    sep="::", 
    header=None, 
    names=MOVIE_COLUMN_NAMES, 
    engine="python"
)

ratings_df = pd.read_csv(
    RATINGS_FILE, 
    sep="::", 
    header=None, 
    names=RATINGS_COLUMN_NAMES, 
    engine="python")


In [30]:
print("movies:", movies_df.shape)
print("ratings:", ratings_df.shape)

movies: (10681, 3)
ratings: (10000054, 4)


In [31]:
movies_df.head(3)

,MovieID,Title,Genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance


In [32]:
ratings_df.head(3)

,UserID,MovieID,Rating,Timestamp
0,1,122,5.0,838985046
1,1,185,5.0,838983525
2,1,231,5.0,838983392


## Ratings stats

In [33]:
print("missing values per column:")
print(ratings_df.isna().mean().sort_values(ascending=False))

missing values per column:
UserID       0.0
MovieID      0.0
Rating       0.0
Timestamp    0.0
dtype: float64


In [35]:
print("rating distribution:")
print(ratings_df["Rating"].value_counts().sort_index())

rating distribution:
Rating
0.5      94988
1.0     384180
1.5     118278
2.0     790306
2.5     370178
3.0    2356676
3.5     879764
4.0    2875850
4.5     585022
5.0    1544812
Name: count, dtype: int64


In [37]:
# timestamps are Unix-seconds in MovieLens (Amazon 2023 was milliseconds)
ts = pd.to_datetime(ratings_df["Timestamp"], unit="s")
print("rating time range:", ts.min(), "to", ts.max())

rating time range: 1995-01-09 11:46:49 to 2009-01-05 05:02:16


In [39]:
dup_pairs = ratings_df.duplicated(subset=["UserID", "MovieID"]).sum()
print(f"duplicated (UserID, MovieID) rows: {dup_pairs} / {len(ratings_df)}")

duplicated (UserID, MovieID) rows: 0 / 10000054


In [41]:
inter_per_user = ratings_df.groupby("UserID").size()
inter_per_item = ratings_df.groupby("MovieID").size()
print("interactions per user:\n", inter_per_user.describe())
print("\ninteractions per item:\n", inter_per_item.describe())

interactions per user:
 count    69878.00000
mean       143.10733
std        216.71258
min         20.00000
25%         35.00000
50%         69.00000
75%        156.00000
max       7359.00000
dtype: float64

interactions per item:
 count    10677.000000
mean       936.597733
std       2487.328304
min          1.000000
25%         34.000000
50%        135.000000
75%        626.000000
max      34864.000000
dtype: float64


## Movies stats

In [42]:
print("missing values per column:")
print(movies_df.isna().mean().sort_values(ascending=False))

missing values per column:
MovieID    0.0
Title      0.0
Genres     0.0
dtype: float64


In [44]:
# year is embedded in the title, e.g. "Toy Story (1995)"; a handful of titles omit it
has_year = movies_df["Title"].str.contains(r"\(\d{4}\)$", regex=True)
print(f"titles with a parenthesized year: {has_year.mean():.1%}")

titles with a parenthesized year: 100.0%


In [46]:
flat_genres = movies_df["Genres"].str.split("|").explode()
print("top genres:")
print(flat_genres.value_counts().head(20))

top genres:
Genres
Drama                 5339
Comedy                3703
Thriller              1706
Romance               1685
Action                1473
Crime                 1118
Adventure             1025
Horror                1013
Sci-Fi                 754
Fantasy                543
Children               528
War                    511
Mystery                509
Documentary            482
Musical                436
Animation              286
Western                275
Film-Noir              148
IMAX                    29
(no genres listed)       1
Name: count, dtype: int64


In [48]:
no_genres = (movies_df["Genres"] == "(no genres listed)").mean()
print(f"movies with no genres listed: {no_genres:.1%}")

movies with no genres listed: 0.0%


## Ratings ↔ movies join coverage

In [50]:
coverage = ratings_df["MovieID"].isin(movies_df["MovieID"]).mean()
print(f"ratings whose MovieID has metadata: {coverage:.1%}")

ratings whose MovieID has metadata: 100.0%
